In [ ]:
import os
from pathlib import Path
import skimage as ski
import tifffile
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt

load_dotenv("../.env")
DATA_ROOT = Path(os.environ.get("DATA_ROOT"))
dir_gt = DATA_ROOT / "processed/labeled/Au_01-vol_01/labeled-01"
dir_save = DATA_ROOT / "figures/segmentation/nanoparticles"
dir_save.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Helvetica"

In [ ]:
file_list = list(dir_gt.glob("*.tif"))

file_name = file_list[0]
image = tifffile.imread(file_name)
# threshold = ski.filters.threshold_otsu(image)
thresholds = ski.filters.threshold_multiotsu(image, 3)
image_th = (image > 200) * 255
# image_th = image_th.astype(np.ubyte)
# imwrite(dir_save / f"{file_name.stem}-nps.tif", image_th)

hist, bin_edges = np.histogram(image, bins=256)
hist = hist / hist.sum()
cdf = np.cumsum(hist)
cdf = cdf / cdf[-1]

fig, ax = plt.subplots()
# ax.hist(image.ravel(), bins=256)

ax.fill_between(bin_edges[:-1], hist, step="mid")
ax2 = ax.twinx()
ax2.plot(bin_edges[:-1], cdf, color="red", label="CDF")

# ax.plot(np.arange(0, 256), hist)
ax.set_title("Histogram")
ax.set(
    # xlim=(180, 255)
    ylim=(0, hist.max() * 1.1)
)
ax2.set(ylim=(0, 1))
ax.axvline(thresholds[-1], color="y")
# for th in thresholds:
#     ax.axvline(th, color='g')